### 💡 Feature Engineering Business Rationale & Assumptions

* [cite_start]**`time_since_signup_hr`**: Calculated as the duration between `signup_time` and `purchase_time`[cite: 115]. 
    * *Business Rationale*: Fraudulent operations utilizing bot orchestration typically process payments immediately following registration. Legitimate users generally exhibit browsing intervals before conversion.
* [cite_start]**`transactions_per_device` (Velocity Parameter)**: Number of unique user interactions occurring on an identical `device_id`[cite: 111].
    * *Business Rationale*: Multiple unique profile accounts attempting rapid checkouts from an identical hardware signature indicates high-risk activity (e.g., device farms or stolen credentials).
* [cite_start]**Scaling Strategy**: We select `StandardScaler` for continuous metrics (`purchase_value`, `age`)[cite: 117].
    * *Assumption*: Preserves variant metrics cleanly for our downstream baseline linear estimators, ensuring high-magnitude attributes do not overshadow features with smaller ranges.
* **Duplicate Criteria**: Complete record row duplication matches are omitted via full vector identification. Shared user accounts or identical values across unique devices are preserved to capture velocity attacks.

# Completed: Latency, frequency, and geo transformations
Advanced Feature Engineering
Here, you extract temporal hooks and velocity features from raw signals. These features help tree-based models find automated bot patterns natively.

Feature Engineering (Fraud_Data.csv)
    Transaction frequency and velocity: number of transactions per user in time windows.
    Time features: 
        hour_of_day, 
        day_of_week.
    time_since_signup: duration between signup_time and purchase_time

In [1]:
import pandas as pd
import numpy as np

# Load Datasets
fraud_df = pd.read_csv("../data/raw/Fraud_Data.csv")
credit_df = pd.read_csv("../data/raw/creditcard.csv")
ip_df = pd.read_csv("../data/raw/IpAddress_to_Country.csv")


In [2]:
# ==========================================
# STEP 0: Clean up the DataFrame Structure
# ==========================================
# Bring purchase_time back from the index into a regular column if it's trapped
if 'purchase_time' not in fraud_df.columns:
    fraud_df = fraud_df.reset_index()

# Ensure the time columns are actual datetime objects (good safety practice)
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])

# Crucial: Sort by time BEFORE creating features or running rolling windows
fraud_df = fraud_df.sort_values('purchase_time').reset_index(drop=True)


# ==========================================
# STEP 1: Time-since-signup feature calculation
# ==========================================
fraud_df['time_since_signup'] = (fraud_df['purchase_time'] - fraud_df['signup_time']).dt.total_seconds() / 3600.0


# ==========================================
# STEP 2: Cyclical Temporal Profiles
# ==========================================
fraud_df['hour_of_day'] = fraud_df['purchase_time'].dt.hour
fraud_df['day_of_week'] = fraud_df['purchase_time'].dt.dayofweek


# ==========================================
# STEP 3: Transaction Velocity (Rolling 24h frequency per device)
# ==========================================
rolling_counts = (
    fraud_df.set_index('purchase_time')
    .groupby('device_id')['device_id']
    .rolling('24h')
    .count()
    .reset_index(level=0, drop=True)  # Removes the group name index level
)

# Because we sorted fraud_df and reset its index in Step 0, 
# rolling_counts.values aligns flawlessly.
fraud_df['device_velocity_24h'] = rolling_counts.values

Data Transformation
    Normalize/scale numerical features (StandardScaler or MinMaxScaler)
    One-hot encode categorical features.

Data Scaling & Categorical Transposition
    Distance-based baselines (like Logistic Regression) can be skewed by large numeric values, such as an e-commerce purchase_value or a banking transaction Amount. We apply standard scaling to normalize these distributions

In [3]:
# print(X_fraud.columns.tolist())

In [4]:
from sklearn.preprocessing import StandardScaler

# 1. Isolate target strings and clear non-predictive system keys
X_fraud = fraud_df.drop(['user_id', 'signup_time', 'purchase_time', 'device_id', 'class'], axis=1, errors='ignore')
y_fraud = fraud_df['class']

# 2. One-Hot Encode Lower Cardinality Columns
categorical_cols = ['source', 'browser', 'sex', 'day_of_week', 'hour_of_day']
X_fraud = pd.get_dummies(X_fraud, columns=categorical_cols, drop_first=True)

# 3. Standardize Numerical Boundaries
num_cols_fraud = ['purchase_value', 'age', 'time_since_signup', 'device_velocity_24h']
scaler_fraud = StandardScaler()
X_fraud[num_cols_fraud] = scaler_fraud.fit_transform(X_fraud[num_cols_fraud])

# 4. Process Bank Dataset separately (Features V1-V28 are already PCA scaled)
X_credit = credit_df.drop(['Class'], axis=1)
y_credit = credit_df['Class']
X_credit[['Time', 'Amount']] = StandardScaler().fit_transform(X_credit[['Time', 'Amount']])

Stratified Splitting & Class Imbalance Resolution
To prevent data leakage, you must split your data before using any resampling techniques. Applying SMOTE to your test or validation sets creates synthetic leakage, leading to artificially inflated performance metrics that won't hold up in production.Justification for SMOTE: Randomly removing legitimate data (undersampling) discards valuable transaction patterns. Instead, SMOTE synthesizes new examples along the line segments joining the $k$-nearest neighbors of existing fraud cases, helping the model learn more robust minority boundaries.

Handle Class Imbalance
    Apply SMOTE or undersampling on the training set only.
    Justify your choice of technique
    Document the class distribution before and after resampling


In [5]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# 1. Execute Strict Stratified Train/Test Splits
X_f_train, X_f_test, y_f_train, y_f_test = train_test_split(
    X_fraud, y_fraud, test_size=0.3, stratify=y_fraud, random_state=42
)

# 2. Apply SMOTE only to the TRAINING set
smote = SMOTE(random_state=42)
X_f_train_resampled, y_f_train_resampled = smote.fit_resample(X_f_train, y_f_train)

print(f"Original training shape: {X_f_train.shape}")
print(f"Resampled training shape: {X_f_train_resampled.shape}")

Original training shape: (105778, 41)
Resampled training shape: (191744, 41)


In [6]:
# !pip install imblearn

In [7]:
# import sys
# !{sys.executable} -m pip install imbalanced-learn